<a href="https://colab.research.google.com/github/427paul/AI_Agent/blob/main/%5BBDA%5D_9%E1%84%8C%E1%85%AE%E1%84%8E%E1%85%A1_GeminiAPI_LangChain_%EA%B3%BC%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -U "langchain-huggingface" "huggingface_hub" wikipedia -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 671.5/671.5 kB 3.8 MB/s eta 0:00:00


In [4]:
import os
def load_api_keys(filepath="api_key.txt"):
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if line and "=" in line:
                key, value = line.split("=", 1)
                os.environ[key.strip()] = value.strip()

path = '/content/drive/MyDrive/LangGraph/'

# API 키 로드 및 환경변수 설정
load_api_keys(path + 'api_key.txt')

In [5]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
llm_ep = HuggingFaceEndpoint(repo_id="openai/gpt-oss-20b", task="conversational")

# HuggingFace에서 가져온 모델을 그대로 쓰지 않고,
# LangChain에서 쉽게 쓰도록 감싸는(wrapper) 단계
llm = ChatHuggingFace(llm=llm_ep)

---
## 학생 실습 ✏️

### 실습 1: Gemini API로 뉴스 카테고리 분류

In [7]:
# ✏️ 실습 1: 뉴스 기사 카테고리 분류기
#
# 요구사항:
# 1. system_instruction으로 역할 설정
# 2. 카테고리: 정치/경제/사회/문화/IT 중 하나
# 3. 아래 3개 기사를 분류하세요

from langchain_core.messages import SystemMessage, HumanMessage

articles = [
    "삼성전자가 새로운 AI 반도체 칩을 발표했다. 성능이 기존 대비 2배 향상됐다.",
    "정부가 내년도 예산안을 국회에 제출했다. 복지 분야 예산이 크게 늘었다.",
    "BTS의 새 앨범이 빌보드 차트 1위를 기록했다.",
]

# ✅ 실습 1 정답
for article in articles:
    # LangChain에서는 SystemMessage와 HumanMessage를 하나의 대화 템플릿으로 묶어 보냅니다.
    messages = [
        SystemMessage(content="뉴스 기사를 정치/경제/사회/문화/IT 중 하나로 분류하세요. 카테고리만 답하세요."),
        HumanMessage(content=f"기사: {article}")
    ]

    # 모델 호출
    response = llm.invoke(messages)
    print(f"  [{response.text.strip()}] {article[:30]}...")

  [IT] 삼성전자가 새로운 AI 반도체 칩을 발표했다. 성능이 ...
  [정치] 정부가 내년도 예산안을 국회에 제출했다. 복지 분야 예...
  [문화] BTS의 새 앨범이 빌보드 차트 1위를 기록했다....


### 실습 2: LangChain으로 리뷰 분석 파이프라인

In [10]:
# ✏️ 실습 2: LangChain LCEL로 리뷰 분석 파이프라인
#
# 요구사항:
# 1. ChatPromptTemplate + ChatGoogleGenerativeAI + JsonOutputParser
# 2. 출력: {"sentiment": "...", "score": 1~5, "keywords": [...]}
# 3. batch로 아래 5개 리뷰를 한 번에 분석

from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

test_reviews = [
    {"review": "배송 빠르고 품질 좋아요 재구매 의사 있습니다"},
    {"review": "불량품이 왔는데 환불이 안되네요 최악"},
    {"review": "가격 대비 그럭저럭 괜찮습니다"},
    {"review": "디자인이 예쁘고 사용감도 좋습니다 만족"},
    {"review": "사이즈가 안 맞아서 교환했는데 한참 걸렸어요"},
]

prompt = ChatPromptTemplate.from_messages([
    ("system", """리뷰를 분석하여 아래 JSON으로만 응답하세요.
{{"sentiment": "긍정/부정/중립", "score": 1~5, "keywords": ["키워드1", "키워드2"]}}"""),
    ("user", "리뷰: {review}"),
])

llm_ep = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-20b",
    task="conversational",
    temperature=0.001
)
llm = ChatHuggingFace(llm=llm_ep)

chain = prompt | llm | JsonOutputParser()

test_reviews = [
    {"review": "배송도 빠르고 제품 퀄리티도 대박이네요! 아주 만족합니다."},
    {"review": "그냥 그래요.. 가격 대비 성능은 잘 모르겠습니다."},
    {"review": "최악입니다. 포장도 다 뜯어져서 오고 작동도 안 해요."}
]

# 4. 배치 실행 및 결과 출력
results = chain.batch(test_reviews)

print("=== 리뷰 분석 결과 ===")
for review, result in zip(test_reviews, results):
    print(f"\n  리뷰: {review['review']}")
    print(f"  분석: {result}")

=== 리뷰 분석 결과 ===

  리뷰: 배송도 빠르고 제품 퀄리티도 대박이네요! 아주 만족합니다.
  분석: {'sentiment': '긍정', 'score': 5, 'keywords': ['배송', '제품 퀄리티']}

  리뷰: 그냥 그래요.. 가격 대비 성능은 잘 모르겠습니다.
  분석: {'sentiment': '중립', 'score': 3, 'keywords': ['가격 대비 성능', '그냥 그래요']}

  리뷰: 최악입니다. 포장도 다 뜯어져서 오고 작동도 안 해요.
  분석: {'sentiment': '부정', 'score': 1, 'keywords': ['포장', '작동', '최악']}


### 실습 3: 다단계 파이프라인

In [11]:
# ✏️ 실습 3: 다단계 파이프라인
#
# [1단계] 리뷰 → 키워드 3개 추출
# [2단계] 키워드 → 개선 제안 1줄

review = "배송은 빠른데 포장이 엉망이고 제품에 스크래치가 있었다"

from langchain_core.output_parsers import StrOutputParser

# 1단계: 키워드 추출
prompt_kw = ChatPromptTemplate.from_messages([
    ("system", "리뷰에서 핵심 키워드 3개를 추출하세요. 쉼표로 구분, 키워드만 답하세요."),
    ("user", "리뷰: {review}"),
])
chain_kw = prompt_kw | llm | StrOutputParser()

# 2단계: 개선 제안
prompt_suggest = ChatPromptTemplate.from_messages([
    ("system", "아래 키워드를 바탕으로 서비스 개선 제안을 한 줄로 작성하세요."),
    ("user", "키워드: {keywords}"),
])
chain_suggest = prompt_suggest | llm | StrOutputParser()

# 실행
keywords = chain_kw.invoke({"review": review})
print(f"[1단계] 키워드: {keywords}")

suggestion = chain_suggest.invoke({"keywords": keywords})
print(f"[2단계] 개선 제안: {suggestion}")

[1단계] 키워드: 배송, 포장, 스크래치
[2단계] 개선 제안: 배송 과정에서 포장을 강화해 스크래치 발생을 최소화하고, 고객 만족도를 높이도록 개선하겠습니다.
